# Bölüm 12 — BÖLÜM 12: BÜYÜK VERİ ANALİTİĞİ VE DAĞITIK MAKİNE ÖĞRENMESİ

**VERİ MADENCİLİĞİ VE MAKİNE ÖĞRENMESİ**  
*Python ile Temel Analitikten Büyük Veri ve Gerçek Zamanlı Sistemlere*

Bu defter, kitabın 12. bölümündeki tüm kod örneklerini içerir. Her hücrenin başlığı kitaptaki alt bölüme karşılık gelir.


In [ ]:
# Bu bölüm için gerekli paketler
!pip install -q Pillow fastapi matplotlib numpy pandas pydantic pyspark requests scipy tensorflow uvicorn


## 12.2. Hadoop Ekosistemi ve MapReduce Mantığı


### Python ile HDFS Etkileşimi: hdfs3 ve PyArrow

`bolum12/12_02_01_python-ile-hdfs-etkilesimi-hdfs3-ve-pyarrow-2.sh`

_Kitap: Kod 12.1_


In [ ]:
hdfs = pafs.HadoopFileSystem(host='namenode-host', port=8020)


### Python ile HDFS Etkileşimi: hdfs3 ve PyArrow

`bolum12/12_02_01_python-ile-hdfs-etkilesimi-hdfs3-ve-pyarrow.py`

_Kitap: Kod 12.2_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: hdfs istemci kütüphanesi
import hdfs                          # pip install hdfs
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

from hdfs import InsecureClient
import pyarrow.fs as pafs
import pandas as pd

# ---- 1. Basit HDFS İstemcisi (hdfs kütüphanesi) ----
client = InsecureClient('http://namenode-host:9870', user='hdfs')

# Dizin listeleme
dosyalar = client.list('/user/data/logs/')
print(f'HDFS dizinindeki dosya sayısı: {len(dosyalar)}')

# Dosya yükleme (local → HDFS)
client.upload('/user/data/logs/access.log',
              '/local/path/access.log',
              overwrite=True)
print("Dosya HDFS'e yüklendi.")

# Dosya indirme (HDFS → local)
client.download('/user/data/logs/access.log',
                '/local/output/access.log',
                overwrite=True)

# Dosya içeriğini okuma
with client.read('/user/data/logs/access.log', encoding='utf-8') as f:
    icerik = f.read()
    print(f'İlk 200 karakter: {icerik[:200]}')

# ---- 2. PyArrow ile HDFS (Parquet format — büyük veri için ideal) ----
# Parquet: sütun tabanlı, sıkıştırılmış, schema bilgili
# 100x daha küçük dosya boyutu ve çok daha hızlı analitik sorgu

# Pandas DataFrame'i Parquet olarak HDFS'e yaz
df_ornek = pd.DataFrame({
    'kullanici_id': range(1000000),
    'yas': [25 + i % 40 for i in range(1000000)],
    'sehir': ['İstanbul', 'Ankara', 'İzmir'] * 333334,
    'harcama': [100.0 + i * 0.01 for i in range(1000000)]
})

import pyarrow as pa
import pyarrow.parquet as pq

# DataFrame → PyArrow Table → HDFS'e Parquet olarak yaz
tablo = pa.Table.from_pandas(df_ornek)
with hdfs.open_output_stream('/user/data/musteriler.parquet') as f:
    pq.write_table(tablo, f)
print(f'1M satır Parquet olarak yazıldı: {df_ornek.memory_usage().sum() / 1e6:.1f} MB RAM')

# HDFS'ten Parquet okuma
with hdfs.open_input_file('/user/data/musteriler.parquet') as f:
    okunan = pq.read_table(f)
df_geri = okunan.to_pandas()
print(f"HDFS'ten okundu: {len(df_geri)} satır, şema: {list(df_geri.columns)}")

# ---- 3. HDFS Dosya Sistemi Yönetimi ----
# Dizin oluşturma
client.makedirs('/user/data/processed/', permission=755)

# Dosya meta verisi
bilgi = client.status('/user/data/logs/access.log')
print(f'Dosya boyutu: {bilgi["length"] / 1e6:.1f} MB')
print(f'Blok boyutu : {bilgi["blockSize"] / 1e6:.0f} MB')
print(f'Replikasyon : {bilgi["replication"]}x')


### Python Uygulaması I: mrjob ile Kelime Sayımı

`bolum12/12_02_02_python-uygulamasi-i-mrjob-ile-kelime-sayimi.py`

_Kitap: Kod 12.3_


In [ ]:
from mrjob.job import MRJob
from mrjob.step import MRStep
import re

KELIME_RE = re.compile(r"[\w']+")

class GelismisKelimeSayimi(MRJob):
    """
    Çok adımlı MapReduce:
    Adım 1: Kelime frekanslarını say
    Adım 2: En sık 10 kelimeyi bul (sıralama için ikinci M-R)
    """

    def steps(self):
        return [
            MRStep(mapper=self.mapper_kelime_say,
                   combiner=self.combiner_toplam,    # Yerel ön-toplama
                   reducer=self.reducer_toplam),
            MRStep(mapper=self.mapper_ters_cevir,
                   reducer=self.reducer_en_siklar),
        ]

    # ---- ADIM 1: Kelime → (kelime, 1) ----
    def mapper_kelime_say(self, _, satir):
        """Her satırı kelimelerine ayır, küçük harfe çevir"""
        for kelime in KELIME_RE.findall(satir.lower()):
            if len(kelime) > 2:  # Çok kısa kelimeleri filtrele
                yield kelime, 1

    # ---- COMBINER: Yerel pre-reduce (ağ trafiğini azaltır) ----
    def combiner_toplam(self, kelime, sayilar):
        yield kelime, sum(sayilar)

    # ---- REDUCER 1: Toplam frekans hesabı ----
    def reducer_toplam(self, kelime, sayilar):
        yield None, (sum(sayilar), kelime)  # None: tek reducer'a topla

    # ---- ADIM 2: (None, (sayi, kelime)) → (sayi, kelime) ----
    def mapper_ters_cevir(self, _, sayi_kelime):
        sayi, kelime = sayi_kelime
        yield -sayi, kelime   # Eksi: büyükten küçüğe sıralama için

    # ---- REDUCER 2: En sık N kelimeyi döndür ----
    def reducer_en_siklar(self, ters_sayi, kelimeler):
        for i, kelime in enumerate(kelimeler):
            if i >= 10:  # İlk 10'u al
                break
            yield kelime, -ters_sayi

if __name__ == '__main__':
    GelismisKelimeSayimi.run()

# ============================================================
# Basit tek adımlı MapReduce (referans amaçlı)
# ============================================================
class BasitKelimeSayimi(MRJob):
    def mapper(self, _, satir):
        for kelime in KELIME_RE.findall(satir):
            yield (kelime.lower(), 1)

    def combiner(self, kelime, sayilar):  # Yerel ön-reduce
        yield (kelime, sum(sayilar))

    def reducer(self, kelime, sayilar):
        yield (kelime, sum(sayilar))

# Hadoop kümesinde çalıştırma:
# python kelime_sayim.py -r hadoop hdfs:///input/buyuk_metin.txt --output-dir=hdfs:///output/

# Yerel modda test:
# python kelime_sayim.py test_metin.txt


### Python Uygulaması II: Saf Python'da MapReduce Simülasyonu

`bolum12/12_02_02_python-uygulamasi-ii-saf-python-da-mapreduce-sim.py`

_Kitap: Kod 12.5_


In [ ]:
from collections import defaultdict
from functools import reduce as py_reduce
from multiprocessing import Pool
import time

# ---- VERİ SİMÜLASYONU ----
def buyuk_metin_uret(satir_sayisi=100_000):
    import random
    kelimeler = ['python', 'büyük', 'veri', 'spark', 'hadoop', 'makine',
                 'öğrenme', 'dağıtık', 'sistem', 'analiz', 'model', 'küme']
    satirlar = []
    for _ in range(satir_sayisi):
        n = random.randint(5, 15)
        satirlar.append(' '.join(random.choices(kelimeler, k=n)))
    return satirlar

# ---- MAP FONKSİYONU ----
def map_fonksiyon(satir):
    """(satir) → list[(kelime, 1)]"""
    return [(kelime.lower(), 1) for kelime in satir.split()]

# ---- SHUFFLE & SORT ----
def shuffle_sort(anahtar_deger_listesi):
    """Tüm (kelime, 1) çiftlerini anahtara göre grupla"""
    gruplama = defaultdict(list)
    for kelime, sayi in anahtar_deger_listesi:
        gruplama[kelime].append(sayi)
    return dict(gruplama)

# ---- REDUCE FONKSİYONU ----
def reduce_fonksiyon(anahtar, degerler):
    """(kelime, [1, 1, 1, ...]) → (kelime, toplam)"""
    return (anahtar, sum(degerler))

# ---- SEKANSİYEL (TEK MAKİNE) ÇALIŞMA ----
def sekansiyel_word_count(satirlar):
    baslangic = time.time()
    # Map aşaması
    tum_cifter = []
    for satir in satirlar:
        tum_cifter.extend(map_fonksiyon(satir))
    # Shuffle & Sort
    gruplu = shuffle_sort(tum_cifter)
    # Reduce aşaması
    sonuclar = {k: reduce_fonksiyon(k, v) for k, v in gruplu.items()}
    with Pool(n_workers) as pool:
        reduce_sonuclar = pool.starmap(reduce_fonksiyon, gruplu.items())

    sure = time.time() - baslangic
    return sonuclar, sure

# ---- PARALELİZE EDİLMİŞ (ÇOK ÇEKIRDEK) ÇALIŞMA ----
def paralel_word_count(satirlar, n_workers=4):
    baslangic = time.time()
    # Veriyi n_workers parçaya böl
    parcalar = [satirlar[i::n_workers] for i in range(n_workers)]

    # Paralel Map (çok çekirdek)
    with Pool(n_workers) as pool:
        # Her chunk'a map_fonksiyon uygula
        sonuclar = pool.map(
            lambda parca: [p for satir in parca for p in map_fonksiyon(satir)],
            parcalar
        )

    # Shuffle & Sort (birleştir)
    tum_cifter = [c for sonuc in sonuclar for c in sonuc]
    gruplu = shuffle_sort(tum_cifter)

    sure = time.time() - baslangic
    return dict(reduce_sonuclar), sure

# ---- DEMO ----
satirlar = buyuk_metin_uret(100_000)
print(f'Toplam satır sayısı: {len(satirlar):,}')
print(f'Toplam kelime tahmini: {sum(len(s.split()) for s in satirlar[:1000]) * 100:,}')

# Sekansiyel çalıştırma
seq_sonuc, seq_sure = sekansiyel_word_count(satirlar)
en_siklar_seq = sorted(seq_sonuc.values(), key=lambda x: -x[1])[:5]
print(f'\nSekansiyel süre : {seq_sure:.3f} sn')
print(f'En sık 5 kelime : {en_siklar_seq}')

# Not: paralel_word_count lambda ile Pool.map uyumsuzluğu olabilir
# Gerçek Hadoop/Spark ortamında bu sorun yoktur
print('\nGerçek dağıtık işlemde Hadoop/Spark bu adımları otonom yönetir.')


### Python Uygulaması: MapReduce'un Disk Darboğazını Simülasyonla Gösterme

`bolum12/12_02_03_python-uygulamasi-mapreduce-un-disk-darbogazini.py`

_Kitap: Kod 12.6_


In [ ]:
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')  # GUI olmayan ortamlar için
import matplotlib.pyplot as plt

# ---- GECIKME SABİTLERİ (nanosaniye cinsinden) ----
LATENCY_RAM_NS  = 100          # RAM: ~100 ns
LATENCY_HDD_NS  = 10_000_000  # HDD: ~10 ms = 10M ns
LATENCY_SSD_NS  = 100_000      # SSD: ~0.1 ms = 100K ns

# Bant genişliği (MB/s)
BW_RAM_MBS = 50_000    # DDR4: ~50 GB/s
BW_HDD_MBS = 200       # HDD:  ~200 MB/s
BW_SSD_MBS = 3_500     # NVMe: ~3.5 GB/s

def sure_hesapla(veri_mb, iterasyon, depolama='hdd'):
    """
    K-Means'in MapReduce (disk) vs Spark (RAM) süresini hesapla
    veri_mb   : Veri seti boyutu MB
    iterasyon : K-Means iterasyon sayısı
    """
    # Disk tabanlı hesap (MapReduce)
    bw = BW_HDD_MBS if depolama == 'hdd' else BW_SSD_MBS
    disk_transfer_sure = veri_mb / bw   # saniye
    islem_sure_iter    = 0.05 * veri_mb / 1024   # 50 ms/GB hesaplama

    # Her iterasyon: 1 okuma + 1 yazma + hesaplama
    hadoop_sure = (disk_transfer_sure * 2 + islem_sure_iter) * iterasyon

    # RAM tabanlı hesap (Spark)
    ram_transfer = veri_mb / BW_RAM_MBS
    ilk_okuma   = disk_transfer_sure   # İlk kez diskten yükle
    spark_sure  = ilk_okuma + (ram_transfer + islem_sure_iter) * iterasyon

    return hadoop_sure, spark_sure

# ---- ANALİZ: Farklı iterasyon sayılarında süre karşılaştırması ----
VERI_MB = 10_000   # 10 GB veri seti
iterasyonlar = list(range(1, 101))  # 1 - 100 iterasyon

hadoop_sureler = []
spark_sureler  = []

for it in iterasyonlar:
    h, s = sure_hesapla(VERI_MB, it, 'hdd')
    hadoop_sureler.append(h / 60)   # dakikaya çevir
    spark_sureler.append(s / 60)

# Sonuçları yazdır
print(f'=== 10 GB Veri, K-Means Süre Karşılaştırması ===')
print(f'{"İterasyon":<15} {"Hadoop (dk)":>12} {"Spark (dk)":>12} {"Hız Kazanımı":>14}')
print('-' * 55)
for it in [1, 5, 10, 20, 50, 100]:
    h_dk = hadoop_sureler[it-1]
    s_dk = spark_sureler[it-1]
    kazanim = h_dk / s_dk if s_dk > 0 else float('inf')
    print(f'{it:<15} {h_dk:>12.1f} {s_dk:>12.2f} {kazanim:>13.0f}×')

# ============================================================
# GERCEK OLCUM: ara sonucu diske yazmak ne kadara mal oluyor?
# ------------------------------------------------------------
# Yukaridaki hesap analitik bir MODELDIR. Asagida ayni iteratif
# hesap iki bicimde GERCEKTEN calistirilip suresi olculur:
#   (a) her iterasyonun ciktisi diske yazilip geri okunur (MapReduce)
#   (b) ara sonuc bellekte tutulur (Spark)
# Mutlak degerler donaniminiza gore degisir; onemli olan EGIMLER ORANI.
# ============================================================
import os, tempfile

def kmeans_adimi(X, merkezler):
    """Tek K-Means iterasyonu: atama + merkez guncelleme."""
    d = ((X[:, None, :] - merkezler[None, :, :]) ** 2).sum(axis=2)
    etiket = d.argmin(axis=1)
    return np.array([X[etiket == k].mean(axis=0) if (etiket == k).any()
                     else merkezler[k] for k in range(len(merkezler))])

def olc(X, k=4, n_iter=20, diske_yaz=False):
    rng = np.random.default_rng(0)
    merkezler = X[rng.choice(len(X), k, replace=False)]
    gecici = os.path.join(tempfile.gettempdir(), "vmml_ara_sonuc.npy")
    sureler, t0 = [], time.perf_counter()
    for _ in range(n_iter):
        merkezler = kmeans_adimi(X, merkezler)
        if diske_yaz:                      # MapReduce: ara sonuc diske yazilir
            with open(gecici, "wb") as f:
                np.save(f, X)
                f.flush()
                os.fsync(f.fileno())       # isletim sistemi onbellegini atla:
                                           # olculen sey gercek disk turu olsun
            X = np.load(gecici)
        sureler.append(time.perf_counter() - t0)
    if os.path.exists(gecici):
        os.remove(gecici)
    return sureler

np.random.seed(0)
# Veri, isletim sistemi sayfa onbellegine tam sigmayacak kadar buyuk secilir;
# aksi halde "diske yazma" gercekte diske hic gitmez ve fark kaybolur.
X_olc = np.random.randn(150_000, 24).astype(np.float32)
N_IT = 20
s_disk = olc(X_olc.copy(), n_iter=N_IT, diske_yaz=True)
s_ram  = olc(X_olc.copy(), n_iter=N_IT, diske_yaz=False)
oran = s_disk[-1] / s_ram[-1]
print(f"\n=== Gercek olcum ({N_IT} iterasyon, {X_olc.nbytes/1e6:.0f} MB) ===")
print(f"Diske yazarak : {s_disk[-1]:.2f} s")
print(f"Bellekte      : {s_ram[-1]:.2f} s")
print(f"Fark          : {oran:.1f}x")

# ---- GRAFIK ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(range(1, N_IT + 1), s_disk, "r-o", markersize=4,
         label=f"Ara sonuc diske ({s_disk[-1]:.2f} s)")
ax1.plot(range(1, N_IT + 1), s_ram, "b-o", markersize=4,
         label=f"Ara sonuc bellekte ({s_ram[-1]:.2f} s)")
ax1.set_xlabel("Iterasyon"); ax1.set_ylabel("Kumulatif sure (s)")
ax1.set_title(f"Olculen: ayni K-Means, iki veri yolu\n{X_olc.nbytes/1e6:.0f} MB, fark {oran:.1f}x")
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

ax2.plot(iterasyonlar, hadoop_sureler, "r--", lw=1.6, label="MapReduce (HDD)")
ax2.plot(iterasyonlar, spark_sureler, "b--", lw=1.6, label="Spark (RAM)")
ax2.set_xlabel("Iterasyon Sayisi (K-Means)"); ax2.set_ylabel("Sure (dakika)")
ax2.set_title("Analitik model: 10 GB veri\n(olcum degil, gecikme sabitlerinden turetilmistir)")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 12.3. Apache Spark Devrimi: Bellek İçi (In-Memory) Veri İşleme


### RDD İşlemleri: Kapsamlı Python Kodu

`bolum12/12_03_02_rdd-islemleri-kapsamli-python-kodu.py`

_Kitap: Kod 12.7_


In [ ]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import time

# ─────────────────────────────────────────────────────────────────────
# SparkContext Yapılandırması
# ─────────────────────────────────────────────────────────────────────

conf = SparkConf() \
    .setAppName("RDD_Demo") \
    .setMaster("local[*]") \
    .set("spark.executor.memory", "2g") \
    .set("spark.driver.memory", "1g") \
    .set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

# ─────────────────────────────────────────────────────────────────────
# 1. Temel RDD Oluşturma Yöntemleri
# ─────────────────────────────────────────────────────────────────────

# Yöntem 1: Python koleksiyonundan parallelize
numbers = sc.parallelize(range(1, 1_000_001), numSlices=8)  # 8 partition
print(f"Partition sayısı: {numbers.getNumPartitions()}")

# Yöntem 2: Dosyadan okuma (her satır bir eleman)
# lines = sc.textFile("hdfs://namenode:9000/data/logs.txt", minPartitions=16)

# Yöntem 3: Çift listeden
pairs = sc.parallelize([("elma", 3), ("armut", 5), ("elma", 2), ("kiraz", 7)])

# ─────────────────────────────────────────────────────────────────────
# 2. Dönüşüm Zincirleri (Lazy – henüz çalışmıyor)
# ─────────────────────────────────────────────────────────────────────

# map + filter: Tek geçişte (pipeline fusion ile optimize edilir)
squared_evens = (numbers
    .filter(lambda x: x % 2 == 0)   # Çift sayıları seç
    .map(lambda x: x ** 2)           # Karelerini al
)
print("Henüz hesaplanmadı – tembel değerlendirme")

# ─────────────────────────────────────────────────────────────────────
# 3. Eylemler – Hesaplama Tetikleyicileri
# ─────────────────────────────────────────────────────────────────────

start = time.time()
total = squared_evens.reduce(lambda a, b: a + b)
elapsed = time.time() - start
print(f"Çift sayıların kareler toplamı: {total:,}  ({elapsed:.2f}s)")

count = squared_evens.count()
print(f"Çift sayı adedi: {count:,}")

sample = squared_evens.take(5)
print(f"İlk 5 eleman: {sample}")

# ─────────────────────────────────────────────────────────────────────
# 4. Çift (Key-Value) RDD İşlemleri
# ─────────────────────────────────────────────────────────────────────

# reduceByKey: groupByKey'den ÇOK DAHA VERİMLİ
# groupByKey tüm veriyi shuffle eder; reduceByKey önce lokal indirgeme yapar
fruit_totals_good = pairs.reduceByKey(lambda a, b: a + b)
fruit_totals_bad  = pairs.groupByKey().mapValues(sum)  # KÖTÜ PRATIK

print("Meyve toplamları (reduceByKey):", fruit_totals_good.collect())

# sortByKey ile sıralama
sorted_fruits = fruit_totals_good.sortByKey(ascending=True)
print("Sıralı:", sorted_fruits.collect())

# countByKey: Eylem – Her anahtar için sayı döner
count_dict = pairs.countByKey()
print("Her anahtarın sayısı:", dict(count_dict))

# ─────────────────────────────────────────────────────────────────────
# 5. Persist / Cache: İteratif Algoritmalar İçin Kritik
# ─────────────────────────────────────────────────────────────────────

from pyspark import StorageLevel

# Büyük veri setini RAM+Disk'e kalıcı sakla
large_rdd = sc.parallelize(range(10_000_000), 16)
large_rdd.persist(StorageLevel.MEMORY_AND_DISK)

# İlk eylem – veri hesaplanıp cache'lenir
t1 = time.time()
print(f"Toplam: {large_rdd.sum():.0f}  ({time.time()-t1:.2f}s – ilk hesaplama)")

# İkinci eylem – cache'den okunur, çok daha hızlı
t2 = time.time()
print(f"Ortalama: {large_rdd.mean():.2f}  ({time.time()-t2:.2f}s – cache'den)")

large_rdd.unpersist()  # Belleği serbest bırak

# ─────────────────────────────────────────────────────────────────────
# 6. Broadcast Değişkenler: Büyük Lookup Tablosunu Her Executor'a Gönder
# ─────────────────────────────────────────────────────────────────────

# Küçük lookup tablosu (örn. ürün kataloğu)
product_names = {1: "Laptop", 2: "Telefon", 3: "Tablet", 4: "Ekran"}

# broadcast: Tüm Executor'lara bir kez gönderilir, tekrar tekrar ağdan çekilmez
broadcast_products = sc.broadcast(product_names)

orders = sc.parallelize([(1, 150), (2, 80), (1, 200), (3, 45), (4, 320)])
enriched = orders.map(
    lambda x: (broadcast_products.value.get(x[0], "Bilinmiyor"), x[1])
)
print("Zenginleştirilmiş siparişler:", enriched.collect())

# ─────────────────────────────────────────────────────────────────────
# 7. Accumulator: Dağıtık Sayaç
# ─────────────────────────────────────────────────────────────────────

error_count = sc.accumulator(0)

def process_line(line):
    global error_count
    if "ERROR" in line:
        error_count.add(1)
    return line

log_lines = sc.parallelize([
    "INFO: User login",
    "ERROR: DB connection failed",
    "INFO: Data processed",
    "ERROR: Timeout occurred",
    "WARN: High memory usage"
])

processed = log_lines.map(process_line)
processed.count()  # Eylemi tetikle
print(f"Toplam hata satırı: {error_count.value}")

sc.stop()


### SparkSession: Modern Giriş Noktası

`bolum12/12_03_03_sparksession-modern-giris-noktasi.py`

_Kitap: Kod 12.8_


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.window import Window
import datetime

# ─────────────────────────────────────────────────────────────────────
# 1. SparkSession Oluşturma ve Yapılandırma
# ─────────────────────────────────────────────────────────────────────

spark = (SparkSession.builder
    .appName("DataFrame_Advanced_Demo")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")     # Küçük veri için 200'den az
    .config("spark.sql.adaptive.enabled", "true")    # AQE (Adaptive Query Execution)
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

# ─────────────────────────────────────────────────────────────────────
# 2. DataFrame Oluşturma Yöntemleri
# ─────────────────────────────────────────────────────────────────────

# Yöntem 1: Python listesinden – küçük test verisi için
data = [
    ("Ayşe",   "Pazarlama", 58000, "İstanbul", 2019),
    ("Mehmet", "Mühendislik", 75000, "Ankara",  2017),
    ("Zeynep", "Pazarlama", 62000, "İzmir",    2020),
    ("Can",    "Mühendislik", 82000, "İstanbul", 2016),
    ("Selin",  "Finans",    69000, "Ankara",   2018),
    ("Burak",  "Mühendislik", 91000, "İstanbul", 2015),
    ("Nisa",   "Finans",    73000, "İzmir",    2017),
]

# Açık şema tanımı – inferSchema'dan daha güvenli
schema = StructType([
    StructField("isim",      StringType(),  nullable=False),
    StructField("departman", StringType(),  nullable=False),
    StructField("maas",      IntegerType(), nullable=False),
    StructField("sehir",     StringType(),  nullable=True),
    StructField("baslama",   IntegerType(), nullable=False),
])

df = spark.createDataFrame(data, schema)

# Yöntem 2: CSV'den – büyük veri için
# df = spark.read.csv("hdfs://cluster/data/calisanlar.csv",
#                     header=True, schema=schema, encoding="UTF-8")

# Yöntem 3: JSON'dan
# df = spark.read.json("s3://bucket/data/logs/*.json")

# Yöntem 4: Parquet'tan (sütunlu format; büyük analizlerde tercih)
# df = spark.read.parquet("hdfs://cluster/data/parquet/")

print(f"DataFrame satır sayısı: {df.count()}")
print(f"Partition sayısı: {df.rdd.getNumPartitions()}")
df.printSchema()
df.show()

# ─────────────────────────────────────────────────────────────────────
# 3. Temel DataFrame İşlemleri
# ─────────────────────────────────────────────────────────────────────

# Sütun seçimi ve yeni sütun ekle
df2 = df.select(
    "isim", "departman", "maas",
    F.col("maas") * 1.15   .alias("maas_artisli"),    # %15 artış
    (2024 - F.col("baslama")).alias("kidem_yili"),
    F.upper(F.col("sehir")).alias("sehir_buyuk"),
)
df2.show()

# Filtreleme
muhendisler = df.filter(
    (F.col("departman") == "Mühendislik") & (F.col("maas") > 80000)
)
print(f"Yüksek maaşlı mühendis: {muhendisler.count()}")

# Gruplama ve Agregasyon
dept_stats = df.groupBy("departman").agg(
    F.count("isim").alias("calisan_sayisi"),
    F.avg("maas").alias("ortalama_maas"),
    F.max("maas").alias("max_maas"),
    F.min("maas").alias("min_maas"),
    F.stddev("maas").alias("maas_std"),
).orderBy("ortalama_maas", ascending=False)
dept_stats.show()

# ─────────────────────────────────────────────────────────────────────
# 4. Window Fonksiyonları (Analitik Fonksiyonlar)
# ─────────────────────────────────────────────────────────────────────

# Departman içinde maaşa göre sıralama
window_spec = Window.partitionBy("departman").orderBy(F.col("maas").desc())

df_ranked = df.withColumn(
    "dept_ici_siralama",
    F.rank().over(window_spec)
).withColumn(
    "dept_ici_maas_payi",
    (F.col("maas") / F.sum("maas").over(Window.partitionBy("departman")) * 100).cast("int"),
)
df_ranked.select("isim", "departman", "maas", "dept_ici_siralama", "dept_ici_maas_payi").show()

# ─────────────────────────────────────────────────────────────────────
# 5. SQL Sorguları
# ─────────────────────────────────────────────────────────────────────

df.createOrReplaceTempView("calisanlar")

result = spark.sql("""
    SELECT
        departman,
        COUNT(*) as calisan,
        ROUND(AVG(maas), 2) as ort_maas,
        MAX(maas) - MIN(maas) as maas_araliği
    FROM calisanlar
    WHERE baslama >= 2016
    GROUP BY departman
    HAVING COUNT(*) >= 2
    ORDER BY ort_maas DESC
""")
result.show()

# ─────────────────────────────────────────────────────────────────────
# 6. Parquet Formatında Kaydetme (Sütunlu depolama; büyük analitik)
# ─────────────────────────────────────────────────────────────────────

# Partition'lı yazma – departmana göre klasör yapısı oluşturur
# df.write.partitionBy("departman").parquet("output/calisanlar_parquet")

# Parquet okuma – yalnızca ilgili partition'lar okunur (partition pruning)
# df_parquet = spark.read.parquet("output/calisanlar_parquet")
# df_parquet.filter(F.col("departman") == "Mühendislik").show()

spark.stop()


## 12.4. PySpark ve MLlib ile Dağıtık Makine Öğrenmesi


### NLP Uygulaması: MLlib ile Metin Sınıflandırması

`bolum12/12_04_03_nlp-uygulamasi-mllib-ile-metin-siniflandirmasi.py`

_Kitap: Kod 12.10_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: bolum12/12_04_03_uygulama-*
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("VM-ML Bolum12")
         .master("local[*]")
         .getOrCreate())
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# ─────────────────────────────────────────────────────────────────────
# TF-IDF Pipeline: Metin → Sayısal Özellik → Model
# ─────────────────────────────────────────────────────────────────────

# Örnek metin verisi (gerçekte Kafka'dan veya HDFS'ten okunur)
text_data = [
    (0, "bu ürün çok harika ve kaliteli tavsiye ederim"),
    (1, "kötü ürün hayal kırıklığı yarattı"),
    (0, "mükemmel teslimat hızlı ve güvenilir"),
    (1, "ürün bozuk geldi iade ettim"),
    (0, "fiyat performans açısından çok iyi"),
    (1, "satıcı yanıltıcı reklam yapıyor"),
]
text_df = spark.createDataFrame(text_data, ["label", "metin"])

# NLP Pipeline
tokenizer = Tokenizer(inputCol="metin", outputCol="kelimeler")

remover = StopWordsRemover(
    inputCol="kelimeler",
    outputCol="temiz_kelimeler",
    stopWords=["ve", "bu", "çok", "bir", "da", "de"]
)

# TF: Kelime Frekansı (hashingTF → seyrek vektör)
hashingTF = HashingTF(
    inputCol="temiz_kelimeler",
    outputCol="tf",
    numFeatures=2**15  # 32768 özellik boyutu
)

# IDF: Ters Doküman Frekansı
idf = IDF(inputCol="tf", outputCol="tfidf")

lr_nlp = LogisticRegression(featuresCol="tfidf", labelCol="label", maxIter=10)

nlp_pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, lr_nlp])
nlp_model = nlp_pipeline.fit(text_df)

# Yeni metinleri sınıflandır
test_texts = [
    (None, "harika bir ürün kesinlikle tavsiye ederim"),
    (None, "çok kötü kalitesiz ürün aldatmaca"),
]
test_text_df = spark.createDataFrame(test_texts, ["label", "metin"])
result = nlp_model.transform(test_text_df)
result.select("metin", "prediction", "probability").show(truncate=False)

spark.stop()


### 12.4.3. Uygulama: Devasa Veri Seti Üzerinde Dağıtık Sınıflandırma

`bolum12/12_04_03_uygulama-devasa-veri-seti-uzerinde-dagitik-sinif.py`

_Kitap: Kod 12.9_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: PySpark tip tanımları
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, FloatType)
spark = (SparkSession.builder
         .appName("VM-ML Bolum12")
         .master("local[*]")
         .getOrCreate())
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler,
    StandardScaler, PCA, ChiSqSelector, Word2Vec
)
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier,
    GBTClassifier, LinearSVC
)
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.clustering import KMeans, BisectingKMeans
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit
from pyspark.ml import Pipeline
import numpy as np

# ─────────────────────────────────────────────────────────────────────
# 1. SparkSession Başlatma (Büyük Küme Yapılandırması)
# ─────────────────────────────────────────────────────────────────────

spark = (SparkSession.builder
    .appName("Distributed_ML_Complete")
    .master("local[*]")  # Üretimde: yarn veya k8s
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "4")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.ml.linalg.vectorSize", "2048")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

# ─────────────────────────────────────────────────────────────────────
# 2. Sentetik Büyük Veri Oluşturma (Gerçekte HDFS/S3'ten okunur)
# ─────────────────────────────────────────────────────────────────────

import random
n_records = 100_000

# Müşteri tıklama davranışı simülasyonu
schema = StructType([
    StructField("user_id",      IntegerType(), False),
    StructField("age",          IntegerType(), True),
    StructField("gender",       StringType(),  True),
    StructField("kategori",     StringType(),  True),
    StructField("sayfa_goruntuleme", IntegerType(), True),
    StructField("oturum_suresi",DoubleType(),  True),
    StructField("onceki_satin_alma", IntegerType(), True),
    StructField("sehir",        StringType(),  True),
    StructField("cihaz",        StringType(),  True),
    StructField("clicked",      IntegerType(), False),   # Hedef değişken
])

kategoriler = ["Elektronik", "Giyim", "Spor", "Kitap", "Kozmetik"]
sehirler    = ["İstanbul", "Ankara", "İzmir", "Bursa", "Antalya"]
cihazlar    = ["mobile", "desktop", "tablet"]

rows = []
for i in range(n_records):
    yas   = random.randint(18, 65)
    cinsiyet = random.choice(["M", "F"])
    kat   = random.choice(kategoriler)
    pgv   = random.randint(1, 50)
    sure  = round(random.uniform(10, 3600), 1)
    onceki= random.randint(0, 20)
    sehir = random.choice(sehirler)
    cihaz = random.choice(cihazlar)
    # Basit kural: genç + mobil + çok sayfa görüntüleme = yüksek tıklama olasılığı
    prob  = (yas < 35) * 0.3 + (cihaz == "mobile") * 0.2 + min(pgv/50, 0.3) + 0.1
    clicked = 1 if random.random() < prob else 0
    rows.append((i, yas, cinsiyet, kat, pgv, sure, onceki, sehir, cihaz, clicked))

df_raw = spark.createDataFrame(rows, schema)
print(f"Toplam kayıt: {df_raw.count():,}")
print(f"Tıklanma oranı: {df_raw.filter(F.col('clicked')==1).count()/n_records:.2%}")

# ─────────────────────────────────────────────────────────────────────
# 3. Keşifsel Veri Analizi (EDA)
# ─────────────────────────────────────────────────────────────────────

print("\n=== Kategorik Dağılım ===")
df_raw.groupBy("kategori").agg(
    F.count("*").alias("toplam"),
    F.mean("clicked").alias("ort_tiklanma"),
    F.mean("oturum_suresi").alias("ort_sure")
).orderBy("ort_tiklanma", ascending=False).show()

print("\n=== Sayısal İstatistikler ===")
df_raw.select("age", "sayfa_goruntuleme", "oturum_suresi", "onceki_satin_alma").describe().show()

# ─────────────────────────────────────────────────────────────────────
# 4. Özellik Mühendisliği
# ─────────────────────────────────────────────────────────────────────

# Yeni özellikler türet
df_featured = df_raw.withColumn(
    "sayfa_sure_orani",
    F.col("sayfa_goruntuleme") / (F.col("oturum_suresi") / 60 + 1)
).withColumn(
    "toplam_ilgi",
    F.col("sayfa_goruntuleme") * F.col("oturum_suresi") / 1000
).withColumn(
    "yas_grubu",
    F.when(F.col("age") < 25, "genc")
     .when(F.col("age") < 40, "orta")
     .otherwise("yasli"),
)

# ─────────────────────────────────────────────────────────────────────
# 5. Eksik Veri Yönetimi (Büyük Veri Ortamında)
# ─────────────────────────────────────────────────────────────────────

# Eksik değerleri sütun ortalaması ile doldur
numeric_cols = ["age", "sayfa_goruntuleme", "oturum_suresi", "onceki_satin_alma"]
fill_means = {col: df_featured.select(F.mean(col)).first()[0]
              for col in numeric_cols}
df_clean = df_featured.fillna(fill_means)
df_clean = df_clean.fillna({"gender": "Bilinmiyor", "sehir": "Diger"})

# ─────────────────────────────────────────────────────────────────────
# 6. Pipeline Aşamaları Tanımla
# ─────────────────────────────────────────────────────────────────────

# 6a. Kategorik değişkenleri sayısala çevir
cat_cols = ["gender", "kategori", "sehir", "cihaz", "yas_grubu"]
indexed_cols = [f"{c}_idx" for c in cat_cols]
encoded_cols = [f"{c}_ohe" for c in cat_cols]

indexers = [
    StringIndexer(inputCol=c, outputCol=ic, handleInvalid="keep")
    for c, ic in zip(cat_cols, indexed_cols)
]

encoders = [
    OneHotEncoder(inputCol=ic, outputCol=ec, dropLast=True)
    for ic, ec in zip(indexed_cols, encoded_cols)
]

# 6b. Tüm özellikleri tek vektörde topla
num_cols = [
    "age", "sayfa_goruntuleme", "oturum_suresi", "onceki_satin_alma",
    "sayfa_sure_orani", "toplam_ilgi"
]
all_feature_cols = num_cols + encoded_cols

assembler = VectorAssembler(
    inputCols=all_feature_cols,
    outputCol="raw_features",
    handleInvalid="keep"
)

# 6c. Standardizasyon
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

# 6d. PCA ile boyut azaltma (isteğe bağlı)
pca = PCA(k=10, inputCol="scaled_features", outputCol="pca_features")

# ─────────────────────────────────────────────────────────────────────
# 7. Hedef Değişkeni Hazırla
# ─────────────────────────────────────────────────────────────────────

label_indexer = StringIndexer(inputCol="clicked", outputCol="label")

# ─────────────────────────────────────────────────────────────────────
# 8. Eğitim/Test Bölme
# ─────────────────────────────────────────────────────────────────────

train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)
train_df.cache()  # İteratif algoritma için cache et

print(f"Eğitim: {train_df.count():,}  |  Test: {test_df.count():,}")

# ─────────────────────────────────────────────────────────────────────
# 9. Model 1: Lojistik Regresyon Pipeline
# ─────────────────────────────────────────────────────────────────────

lr = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="clicked",
    maxIter=20,
    regParam=0.01,      # L2 regularizasyon
    elasticNetParam=0.0 # 0=L2, 1=L1
)

lr_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, lr])

import time
start = time.time()
lr_model = lr_pipeline.fit(train_df)
print(f"Lojistik Regresyon eğitim süresi: {time.time()-start:.1f}s")

# Değerlendirme
lr_preds = lr_model.transform(test_df)
evaluator_bin = BinaryClassificationEvaluator(
    labelCol="clicked", metricName="areaUnderROC")
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="clicked", predictionCol="prediction", metricName="accuracy")

lr_auc = evaluator_bin.evaluate(lr_preds)
lr_acc = evaluator_acc.evaluate(lr_preds)
print(f"Lojistik Regresyon – AUC: {lr_auc:.4f}  |  Accuracy: {lr_acc:.4f}")

# ─────────────────────────────────────────────────────────────────────
# 10. Model 2: Random Forest
# ─────────────────────────────────────────────────────────────────────

rf = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="clicked",
    numTrees=50,
    maxDepth=8,
    seed=42
)

rf_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, rf])

start = time.time()
rf_model = rf_pipeline.fit(train_df)
print(f"Random Forest eğitim süresi: {time.time()-start:.1f}s")

rf_preds = rf_model.transform(test_df)
rf_auc   = evaluator_bin.evaluate(rf_preds)
rf_acc   = evaluator_acc.evaluate(rf_preds)
print(f"Random Forest – AUC: {rf_auc:.4f}  |  Accuracy: {rf_acc:.4f}")

# Özellik Önem Skoru
rf_fit  = rf_model.stages[-1]
importances = rf_fit.featureImportances
print(f"En önemli özellik indeksleri (ilk 5): {importances.indices[:5].tolist()}")

# ─────────────────────────────────────────────────────────────────────
# 11. Hiperparametre Araması: CrossValidator
# ─────────────────────────────────────────────────────────────────────

lr_cv = LogisticRegression(featuresCol="scaled_features", labelCol="clicked")
cv_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, lr_cv])

param_grid = (ParamGridBuilder()
    .addGrid(lr_cv.maxIter,   [10, 20])
    .addGrid(lr_cv.regParam,  [0.001, 0.01, 0.1])
    .addGrid(lr_cv.elasticNetParam, [0.0, 0.5])
    .build())
print(f"Test edilecek kombinasyon sayısı: {len(param_grid)} (2×3×2 = 12)")

cv = CrossValidator(
    estimator=cv_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=BinaryClassificationEvaluator(labelCol="clicked"),
    numFolds=3,
    seed=42,
    parallelism=4   # Aynı anda 4 kombinasyonu paralel test et
)

print("CrossValidator çalışıyor (dağıtık hiperparametre araması)...")
start = time.time()
cv_model = cv.fit(train_df)
print(f"CrossValidator süresi: {time.time()-start:.1f}s")

cv_preds = cv_model.transform(test_df)
cv_auc   = evaluator_bin.evaluate(cv_preds)
print(f"En iyi model AUC: {cv_auc:.4f}")

# En iyi parametreler
best_lr = cv_model.bestModel.stages[-1]
print(f"En iyi maxIter: {best_lr.getMaxIter()}")
print(f"En iyi regParam: {best_lr.getRegParam()}")

# ─────────────────────────────────────────────────────────────────────
# 12. K-Means Kümeleme (Gözetimsiz)
# ─────────────────────────────────────────────────────────────────────

kmeans = KMeans(featuresCol="scaled_features", k=4, seed=42, maxIter=20)
km_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, kmeans])
km_model = km_pipeline.fit(df_clean)
km_preds = km_model.transform(df_clean)

# Küme profili
km_preds.groupBy("prediction").agg(
    F.count("*").alias("kume_boyutu"),
    F.mean("age").alias("ort_yas"),
    F.mean("clicked").alias("tiklanma_orani"),
    F.mean("oturum_suresi").alias("ort_sure")
).orderBy("kume_boyutu", ascending=False).show()

# Küme içi varyans toplamı (WSSSE)
wssse = km_model.stages[-1].summary.trainingCost
print(f"K-Means WSSSE (küme kalitesi): {wssse:,.2f}")

# ─────────────────────────────────────────────────────────────────────
# 13. ALS ile Öneri Sistemi
# ─────────────────────────────────────────────────────────────────────

# Kullanıcı-ürün etkileşim matrisi simülasyonu
ratings_data = [(int(i % 1000), int(j), float(random.randint(1, 5)))
                for i in range(10000)
                for j in random.sample(range(500), 5)]

ratings_schema = StructType([
    StructField("userId",    IntegerType(), False),
    StructField("productId", IntegerType(), False),
    StructField("rating",    FloatType(),   False),
])

ratings_df = spark.createDataFrame(ratings_data, ratings_schema)

als = ALS(
    maxIter=10,
    regParam=0.1,
    rank=20,
    userCol="userId",
    itemCol="productId",
    ratingCol="rating",
    coldStartStrategy="drop"  # Yeni kullanıcı/ürün için NaN önle
)

train_r, test_r = ratings_df.randomSplit([0.8, 0.2])
als_model = als.fit(train_r)
als_preds = als_model.transform(test_r)

from pyspark.ml.evaluation import RegressionEvaluator
rmse = RegressionEvaluator(metricName="rmse", labelCol="rating",
                            predictionCol="prediction").evaluate(als_preds)
print(f"ALS RMSE: {rmse:.4f}")

# Kullanıcı 1 için en iyi 10 öneri
user_recs = als_model.recommendForAllUsers(10)
user_recs.filter(F.col("userId") == 1).show(truncate=False)

# ─────────────────────────────────────────────────────────────────────
# 14. Model Kaydetme ve Yükleme
# ─────────────────────────────────────────────────────────────────────

# Tüm pipeline'ı HDFS veya yerel diske kaydet
# rf_model.write().overwrite().save("hdfs://cluster/models/rf_ctr_model")
rf_model.write().overwrite().save("/tmp/rf_ctr_model")
print("Model /tmp/rf_ctr_model konumuna kaydedildi.")

# Modeli geri yükle (başka bir Spark uygulamasında)
from pyspark.ml import PipelineModel
loaded_model = PipelineModel.load("/tmp/rf_ctr_model")

# Yeni veriler üzerinde tahmin
new_preds = loaded_model.transform(test_df)
print(f"Yüklenen model AUC: {evaluator_bin.evaluate(new_preds):.4f}")

train_df.unpersist()
spark.stop()


## 12.5. Derin Öğrenme Modellerini Ölçeklendirme ve Dağıtım (Deployment)


### TensorFlow Dağıtık Eğitim: Kapsamlı Python Kodu

`bolum12/12_05_01_tensorflow-dagitik-egitim-kapsamli-python-kodu.py`

_Kitap: Kod 12.11_


In [ ]:
import tensorflow as tf
import numpy as np
import os
import json
import time

# ─────────────────────────────────────────────────────────────────────
# 1. MirroredStrategy: Tek Makine, Çok GPU
# ─────────────────────────────────────────────────────────────────────

strategy = tf.distribute.MirroredStrategy()
print(f"Kullanılabilir GPU sayısı: {strategy.num_replicas_in_sync}")

# Model ve derleme MUTLAKA strategy.scope() içinde olmalı
with strategy.scope():
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False  # Transfer learning: önce dondurup eğit

    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(256, activation="relu",
                               kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

    # Doğrusal ölçekleme: n_gpu × base_lr
    n_gpu  = max(1, strategy.num_replicas_in_sync)
    base_lr = 0.001
    lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=base_lr * n_gpu,
        first_decay_steps=1000,
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3)]
    )

model.summary()

# ─────────────────────────────────────────────────────────────────────
# 2. tf.data.Dataset ile Verimli Dağıtık Veri Yükleme
# ─────────────────────────────────────────────────────────────────────

BATCH_SIZE_PER_REPLICA = 32
GLOBAL_BATCH_SIZE      = BATCH_SIZE_PER_REPLICA * n_gpu
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, [224, 224])
    return image, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    return image, label

(X_tr, y_tr), (X_te, y_te) = tf.keras.datasets.cifar10.load_data()
y_tr, y_te = y_tr.squeeze(), y_te.squeeze()

train_dataset = (tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
    .shuffle(50000, seed=42)
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .batch(GLOBAL_BATCH_SIZE, drop_remainder=True)
    .prefetch(AUTOTUNE)             # GPU çalışırken CPU'da batch hazırla
    .cache()                        # İlk epoch'tan sonra RAM'de sakla
)

test_dataset = (tf.data.Dataset.from_tensor_slices((X_te, y_te))
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(GLOBAL_BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────
# 3. Callback Yapılandırması
# ─────────────────────────────────────────────────────────────────────

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10,
                                      restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir="./logs", histogram_freq=1, update_freq="epoch"
    ),
    tf.keras.callbacks.CSVLogger("training_log.csv"),
]

# ─────────────────────────────────────────────────────────────────────
# 4. Eğitim (feature extraction aşaması)
# ─────────────────────────────────────────────────────────────────────

print("\nAşama 1: Feature Extraction (base model dondurulmuş)")
history_fe = model.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset,
    callbacks=callbacks,
    verbose=1
)

# Fine-tuning: En üst katmanları aç
with strategy.scope():
    base_model.trainable = True
    # Yalnızca son 50 katmanı eğit
    for layer in base_model.layers[:-50]:
        layer.trainable = False
    # Çok daha küçük LR ile fine-tune
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

print("\nAşama 2: Fine-Tuning (son 50 katman açık)")
history_ft = model.fit(
    train_dataset,
    epochs=20,
    validation_data=test_dataset,
    callbacks=callbacks,
    verbose=1
)

# ─────────────────────────────────────────────────────────────────────
# 5. MultiWorkerMirroredStrategy (Çok Makine)
# ─────────────────────────────────────────────────────────────────────

# Her makinede bu ortam değişkeni ayarlanmalı (YAML/JSON):
# TF_CONFIG = {
#   "cluster": {"worker": ["host1:12345", "host2:23456"]},
#   "task": {"type": "worker", "index": 0}   # host1 için 0, host2 için 1
# }
# os.environ["TF_CONFIG"] = json.dumps(TF_CONFIG)
# strategy = tf.distribute.MultiWorkerMirroredStrategy()

print("\nModel başarıyla eğitildi.")


### Docker ile Konteynerizasyon

`bolum12/12_05_02_docker-ile-konteynerizasyon.py`

_Kitap: Kod 12.13_


In [ ]:
import tensorflow as tf
import subprocess
import requests
import json
import numpy as np

# Model SavedModel formatında kaydet (TF Serving için zorunlu)
model = tf.keras.models.load_model("best_model.keras")
model.save("models/cifar10/1")   # Sürüm klasörü: 1, 2, 3...
print("Model SavedModel formatında kaydedildi.")

# ─────────────────────────────────────────────────────────────────────
# docker run -d --name tfserving \
#   -p 8501:8501 \
#   -v /path/to/models:/models \
#   tensorflow/serving \
#   --model_config_file=/models/models.config
# ─────────────────────────────────────────────────────────────────────

# TF Serving REST API test
def predict_via_tfserving(images: np.ndarray, server="localhost:8501") -> dict:
    """TF Serving REST API üzerinden tahmin."""
    url = f"http://{server}/v1/models/cifar10:predict"
    payload = {"instances": images.tolist()}
    response = requests.post(url, json=payload, timeout=10)
    response.raise_for_status()
    return response.json()

# Örnek istek
test_input = np.random.rand(1, 32, 32, 3).astype("float32")
# result = predict_via_tfserving(test_input)
# print(f"TF Serving yanıtı: {result}")

# ─────────────────────────────────────────────────────────────────────
# A/B Test Simülasyonu: Model v1 vs v2
# ─────────────────────────────────────────────────────────────────────

import random

def ab_test_predict(features: np.ndarray, traffic_split_v2: float = 0.1):
    """
    Canary deployment: %10 trafiği v2'ye yönlendir, %90 v1'e.
    Gerçek sistemde bu Istio veya Nginx ile ağ katmanında yapılır.
    """
    if random.random() < traffic_split_v2:
        model_version = "v2"
        # result = predict_via_tfserving(features, "v2-server:8501")
    else:
        model_version = "v1"
        # result = predict_via_tfserving(features, "v1-server:8501")

    # Metrikleri kaydet (MLflow veya Prometheus)
    # metrics_logger.log({"version": model_version, "latency": latency})
    return model_version

print("API, TF Serving ve A/B test altyapısı hazır.")

# ─────────────────────────────────────────────────────────────────────
# Model İzleme (Monitoring): Veri Kayması (Data Drift) Tespiti
# ─────────────────────────────────────────────────────────────────────

from scipy import stats

def detect_data_drift(reference_data: np.ndarray,
                      current_data:   np.ndarray,
                      threshold: float = 0.05) -> dict:
    """
    KS-testi ile veri kayması tespiti.
    p-değeri < threshold ise kayma var demektir.
    """
    drift_report = {}
    for i in range(reference_data.shape[1]):
        ks_stat, p_val = stats.ks_2samp(reference_data[:, i], current_data[:, i])
        drift_report[f"ozellik_{i}"] = {
            "ks_istatistigi": round(ks_stat, 4),
            "p_degeri":       round(p_val, 4),
            "drift_var":      p_val < threshold
        }
    drifted = sum(1 for v in drift_report.values() if v["drift_var"])
    drift_report["ozet"] = {
        "toplam_ozellik": reference_data.shape[1],
        "drift_ozellik":  drifted,
        "drift_orani":    round(drifted / reference_data.shape[1], 3)
    }
    return drift_report

# Örnek kullanım
ref  = np.random.randn(1000, 20)   # Eğitim verisi dağılımı
curr = np.random.randn(200, 20)    # Üretim verisi (normal)
drifted_curr = ref + np.random.randn(200, 20) * 3  # Yüksek drift

report_normal  = detect_data_drift(ref, curr)
report_drifted = detect_data_drift(ref, drifted_curr)

print(f"Normal veri  – Drift eden özellik: {report_normal['ozet']['drift_ozellik']}")
print(f"Drifted veri – Drift eden özellik: {report_drifted['ozet']['drift_ozellik']}")


### FastAPI ile Model REST API'si Oluşturma

`bolum12/12_05_02_fastapi-ile-model-rest-api-si-olusturma.py`

_Kitap: Kod 12.12_


In [ ]:
from fastapi import FastAPI, HTTPException, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager
import tensorflow as tf
import numpy as np
from PIL import Image
import io
import time
import logging
from typing import List, Optional

# ─────────────────────────────────────────────────────────────────────
# Uygulama Yaşam Döngüsü: Model Başlangıçta Yüklenir
# ─────────────────────────────────────────────────────────────────────

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Global model değişkeni
model_registry = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    """FastAPI başladığında modeli yükle; kapanırken temizle."""
    logger.info("Model yükleniyor...")
    try:
        model_registry["cifar10"] = tf.keras.models.load_model("best_model.keras")
        logger.info(f"Model başarıyla yüklendi: {model_registry['cifar10'].name}")
    except Exception as e:
        logger.error(f"Model yüklenemedi: {e}")
        raise
    yield   # Burada uygulama çalışır
    model_registry.clear()
    logger.info("Model bellekten temizlendi.")

app = FastAPI(
    title="CIFAR-10 Görüntü Sınıflandırma API",
    description="ResNet50 tabanlı derin öğrenme modeli ile görüntü sınıflandırma",
    version="1.0.0",
    lifespan=lifespan
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ─────────────────────────────────────────────────────────────────────
# Veri Modelleri (Pydantic)
# ─────────────────────────────────────────────────────────────────────

CIFAR10_CLASSES = ["uçak", "otomobil", "kuş", "kedi", "geyik",
                   "köpek", "kurbağa", "at", "gemi", "kamyon"]

class Prediction(BaseModel):
    sinif: str = Field(..., description="Tahmin edilen sınıf adı")
    sinif_id: int = Field(..., description="Sınıf indeksi")
    guven: float = Field(..., description="Güven skoru (0-1)")
    tum_olasiliklar: List[float] = Field(..., description="Tüm sınıfların olasılıkları")
    gecikme_ms: float = Field(..., description="Çıkarım süresi (milisaniye)")

class HealthResponse(BaseModel):
    durum: str
    model_yuklu: bool
    tensorflow_surumu: str

# ─────────────────────────────────────────────────────────────────────
# API Endpoint'leri
# ─────────────────────────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse, tags=["Sistem"])
async def health_check():
    """Servis sağlık kontrolü."""
    return HealthResponse(
        durum="aktif",
        model_yuklu="cifar10" in model_registry,
        tensorflow_surumu=tf.__version__
    )

@app.post("/predict/image", response_model=Prediction, tags=["Tahmin"])
async def predict_image(file: UploadFile = File(...)):
    """
    Görüntü dosyası yükleyerek sınıflandırma yap.
    Desteklenen formatlar: JPEG, PNG, BMP
    """
    if "cifar10" not in model_registry:
        raise HTTPException(status_code=503, detail="Model henüz hazır değil")

    # Dosya türü kontrolü
    if file.content_type not in ["image/jpeg", "image/png", "image/bmp"]:
        raise HTTPException(status_code=400,
            detail=f"Desteklenmeyen dosya türü: {file.content_type}")

    try:
        # Görüntüyü oku ve önişle
        content = await file.read()
        image   = Image.open(io.BytesIO(content)).convert("RGB")
        image   = image.resize((32, 32))  # CIFAR-10 boyutu
        img_array = np.array(image) / 255.0
        img_array = np.expand_dims(img_array, axis=0)  # (1, 32, 32, 3)

        # Tahmin
        model = model_registry["cifar10"]
        start = time.perf_counter()
        probs = model.predict(img_array, verbose=0)[0]
        elapsed_ms = (time.perf_counter() - start) * 1000

        best_idx = int(np.argmax(probs))
        return Prediction(
            sinif=CIFAR10_CLASSES[best_idx],
            sinif_id=best_idx,
            guven=float(probs[best_idx]),
            tum_olasiliklar=[float(p) for p in probs],
            gecikme_ms=round(elapsed_ms, 2)
        )
    except Exception as e:
        logger.error(f"Tahmin hatası: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict/batch", tags=["Tahmin"])
async def predict_batch(files: List[UploadFile] = File(...)):
    """Çoklu görüntü için batch tahmin."""
    if len(files) > 32:
        raise HTTPException(400, "Tek seferde en fazla 32 görüntü")

    results = []
    for f in files:
        try:
            pred = await predict_image(f)
            results.append({"dosya": f.filename, "tahmin": pred})
        except Exception as e:
            results.append({"dosya": f.filename, "hata": str(e)})
    return {"sonuclar": results, "toplam": len(results)}

# Çalıştırma: uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
